In [1]:
!pip install -r "C:/Users/Lenovo/Desktop/Lorenzo/Code/Training/notebooks/requirements_py311-cuda.txt"


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
# 1. Manually add the site-packages folder of your environment to the path
sys.path.append(r"C:\Users\Lenovo\Desktop\Lorenzo\habs_test_311\Lib\site-packages")

# 2. Now try the import
import torch
import pandas as pd

print(f"Torch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Torch Version: 2.5.1+cu121
CUDA Available: True


## Section 1: Import Required Libraries and Check CUDA Setup

Import all necessary libraries and verify GPU/CUDA availability.

In [4]:
# Import Required Libraries
from dl_funcs_notebook_gpu import (
    eval_process_folders, eval_create_dataloaders, eval_get_evals, eval_grad_cam
)
import torch
import os
import warnings
import numpy as np
import matplotlib
import pandas as pd
from datetime import datetime, timedelta
from ast import literal_eval

# Configure matplotlib and suppress warnings
matplotlib.use('agg')  # Use non-interactive backend
warnings.filterwarnings('ignore')

# ============================================================================
# CUDA/GPU INITIALIZATION AND CONFIGURATION
# ============================================================================

print("="*80)
print("CUDA DEVICE SETUP")
print("="*80)

# Check CUDA availability
cuda_available = torch.cuda.is_available()
cudnn_enabled = torch.backends.cudnn.enabled

print(f"CUDA Available: {cuda_available}")
print(f"CUDA Device Count: {torch.cuda.device_count() if cuda_available else 0}")
print(f"cuDNN Enabled: {cudnn_enabled}")

if cuda_available:
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")
    
    # Get GPU memory information
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU Memory: {gpu_memory:.2f} GB")
    
    # Optimize cuDNN for performance
    torch.backends.cudnn.benchmark = True
    print("\nGPU optimization enabled:")
    print(f"  - cuDNN Benchmark: {torch.backends.cudnn.benchmark}")
    print(f"  - Deterministic: {torch.backends.cudnn.deterministic}")
else:
    print("\nWARNING: CUDA is not available. Falling back to CPU.")
    print("For GPU acceleration, ensure nvidia-cudnn-cu12 is installed.")

# Select device (GPU if available, CPU otherwise)
device = torch.device("cuda:0" if cuda_available else "cpu")
print(f"\nSelected Device: {device}")
print("="*80)

CUDA DEVICE SETUP
CUDA Available: True
CUDA Device Count: 1
cuDNN Enabled: True
CUDA Version: 12.1
Device Name: NVIDIA GeForce RTX 4060 Laptop GPU
Device Capability: (8, 9)
Total GPU Memory: 8.59 GB

GPU optimization enabled:
  - cuDNN Benchmark: True
  - Deterministic: False

Selected Device: cuda:0


## Section 2: Configuration and Setup

Define paths, parameters, and reference mappings for the evaluation workflow.

In [5]:
# Configuration and Setup

# Define paths
reference_path = "C:/Users/Lenovo/Desktop/Lorenzo/Test/reference_list.txt"
source_dir = "C:/Users/Lenovo/Desktop/Lorenzo/Test/Fixed_Preserved"
export_dir = "C:/Users/Lenovo/Desktop/Lorenzo/Test/res"
model_src = "C:/Users/Lenovo/Desktop/Lorenzo/Code/Models/Fixed/models"
index_file = "C:/Users/Lenovo/Desktop/Lorenzo/Test/Fixed_feat.csv"
data_pkl_path = "C:/Users/Lenovo/Desktop/Lorenzo/Test/data_dl.pkl"
output_base_dir = "C:/Users/Lenovo/Desktop/Lorenzo/Test/res"

# Parameters
num_samples = 250
watch_list = ['Fixed_Preserved', 'Others']
watch_list.sort()

# GPU batch size (larger batch sizes for GPU processing)
batch_size = 32 if cuda_available else 16  # Larger batches for GPU
print(f"Batch size: {batch_size}")

# Load reference mapping
reference = {}
with open(reference_path, "r") as f:
    file_contents = f.readlines()
    for line in file_contents:
        temp = line.split(":", 2)
        if len(temp) == 2:
            reference[str(temp[1]).strip()] = int(temp[0])

watch_list = list(reference.keys())
print(f"Reference mapping: {reference}")
print(f"Watch list (classes): {watch_list}")

# Helper function to prepare output folders
def prepareOutputFolder(exp_dir):
    if not os.path.exists(exp_dir):
        os.makedirs(exp_dir, exist_ok=True)

print("Configuration loaded successfully!")

Batch size: 32
Reference mapping: {'Chaetoceros': 0, 'Others': 1}
Watch list (classes): ['Chaetoceros', 'Others']
Configuration loaded successfully!


## Section 3: Data Loading and GPU Memory Management

Load or generate the dataset from the source folder with GPU-aware memory management.

In [6]:
# Data Loading with GPU Memory Optimization

data = pd.DataFrame()

if not os.path.exists(data_pkl_path):
    print("No saved copy of dataset found, generating from source folder...")
    data = eval_process_folders(source_dir, reference)
else:
    print("A saved copy of the dataset was found, loading...")
    data = pd.read_pickle(data_pkl_path)

print(f"Dataset loaded. Total records: {len(data)}")
print(f"Dataset shape: {data.shape}")
print(f"Dataset columns: {data.columns.tolist()}")
print(f"\nFirst few rows:")
print(data.head())

# Clear any GPU cache to prepare for model evaluation
if cuda_available:
    torch.cuda.empty_cache()
    print(f"\nGPU Memory after data loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

A saved copy of the dataset was found, loading...
Dataset loaded. Total records: 70
Dataset shape: (70, 8)
Dataset columns: ['image', 'filename', 'roi_number', 'label', 'volume_analyzed', 'humidity', 'temperature', 'image_size']

First few rows:
                                               image  \
0  [[0.8196078431372549, 0.8196078431372549, 0.80...   
1  [[0.8117647058823529, 0.8235294117647058, 0.8,...   
2  [[0.792156862745098, 0.7843137254901961, 0.8, ...   
3  [[0.792156862745098, 0.7725490196078432, 0.788...   
4  [[0.7764705882352941, 0.7843137254901961, 0.80...   

                             filename  roi_number  label  volume_analyzed  \
0  D20250609T054352_IFCB202_10_10.png          -1      1               -1   
1  D20250609T054352_IFCB202_11_11.png          -1      1               -1   
2  D20250609T054352_IFCB202_12_12.png          -1      1               -1   
3  D20250609T054352_IFCB202_13_13.png          -1      1               -1   
4  D20250609T054352_IFCB202_14_1

## Section 4: Evaluation Execution (Split Loop) with GPU Acceleration

Run the 5-split cross-validation with GPU-optimized processing.

In [7]:
# Initialize results tracking
results_summary = {}
all_predictions = {}
all_ground_truth = {}

# Run evaluation for splits 1-5
for split_num in range(1, 6):
    print(f"\n{'='*80}")
    print(f"SPLIT {split_num}")
    print(f"{'='*80}")
    
    try:
        # Create dataloaders with GPU batch size
        print(f"Creating dataloaders for split {split_num} (batch_size={batch_size})...")
        train_loader, val_loader, test_loader = eval_create_dataloaders(
            data, split_num, batch_size=batch_size
        )
        
        # Run evaluation
        print(f"Running model evaluation for split {split_num} on {device}...")
        val_metrics, test_metrics, all_preds, all_true = eval_get_evals(
            split_num, val_loader, test_loader
        )
        
        # Store results
        results_summary[f'split_{split_num}'] = {
            'val_metrics': val_metrics,
            'test_metrics': test_metrics
        }
        all_predictions[f'split_{split_num}'] = all_preds
        all_ground_truth[f'split_{split_num}'] = all_true
        
        # Export results to CSV
        results_dir = os.path.join(output_base_dir, f'split_{split_num}')
        os.makedirs(results_dir, exist_ok=True)
        
        # Save evaluation metrics
        metrics_df = pd.DataFrame({
            'Metric': list(test_metrics.keys()),
            'Test_Value': list(test_metrics.values())
        })
        metrics_df.to_csv(os.path.join(results_dir, f'eval_metrics_split_{split_num}.csv'), index=False)
        
        # Save predictions
        preds_df = pd.DataFrame({
            'Predictions': all_preds,
            'Ground_Truth': all_true
        })
        preds_df.to_csv(os.path.join(results_dir, f'predictions_split_{split_num}.csv'), index=False)
        
        print(f"Results exported to {results_dir}")
        print(f"Test Metrics: {test_metrics}")
        
        # GPU memory cleanup
        if cuda_available:
            torch.cuda.empty_cache()
            print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
            
    except Exception as e:
        print(f"Error processing split {split_num}: {str(e)}")
        if cuda_available:
            torch.cuda.empty_cache()
        continue

print(f"\n{'='*80}")
print("All splits evaluated successfully!")
print(f"{'='*80}")


SPLIT 1
Creating dataloaders for split 1 (batch_size=32)...
Running model evaluation for split 1 on cuda:0...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_1
Test Metrics: {'accuracy': 0.82, 'precision': 0.81, 'recall': 0.82}
GPU Memory: 0.00 GB

SPLIT 2
Creating dataloaders for split 2 (batch_size=32)...
Running model evaluation for split 2 on cuda:0...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_2
Test Metrics: {'accuracy': 0.82, 'precision': 0.81, 'recall': 0.82}
GPU Memory: 0.00 GB

SPLIT 3
Creating dataloaders for split 3 (batch_size=32)...
Running model evaluation for split 3 on cuda:0...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_3
Test Metrics: {'accuracy': 0.82, 'precision': 0.81, 'recall': 0.82}
GPU Memory: 0.00 GB

SPLIT 4
Creating dataloaders for split 4 (batch_size=32)...
Running model evaluation for split 4 on cuda:0...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_4
Test Metrics: {'

## Section 5: Grad-CAM Visualization with GPU Acceleration

Generate Grad-CAM visualizations with GPU-accelerated processing.

In [8]:
# Generate Grad-CAM visualizations for each split with GPU optimization
print(f"\n{'='*80}")
print("GRAD-CAM VISUALIZATION GENERATION")
print(f"{'='*80}")

for split_num in range(1, 6):
    print(f"\nGenerating Grad-CAM visualizations for split {split_num}...")
    
    try:
        # Create dataloaders again for Grad-CAM
        train_loader, val_loader, test_loader = eval_create_dataloaders(
            data, split_num, batch_size=batch_size
        )
        
        # Run Grad-CAM with GPU acceleration
        eval_grad_cam(
            split_num, test_loader,
            reference=reference,
            device=device,
            export_tree=output_base_dir,
            model_source=model_src,
            num_samples=num_samples
        )
        
        print(f"Grad-CAM visualizations saved for split {split_num}")
        
        # Clean up GPU memory
        if cuda_available:
            torch.cuda.empty_cache()
            print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
            
    except Exception as e:
        print(f"Error generating Grad-CAM for split {split_num}: {str(e)}")
        if cuda_available:
            torch.cuda.empty_cache()
        continue

print(f"\nGrad-CAM generation complete for all splits!")


GRAD-CAM VISUALIZATION GENERATION

Generating Grad-CAM visualizations for split 1...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_1\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:06<00:00,  1.40s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:04<00:00,  1.38s/it]


Grad-CAM visualizations saved for split 1
GPU Memory: 0.00 GB

Generating Grad-CAM visualizations for split 2...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_2\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:08<00:00,  1.47s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:04<00:00,  1.62s/it]


Grad-CAM visualizations saved for split 2
GPU Memory: 0.00 GB

Generating Grad-CAM visualizations for split 3...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_3\grad_cam


...

Grad-CAM visualizations saved for split 3
GPU Memory: 0.00 GB

Generating Grad-CAM visualizations for split 4...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_4\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:05<00:00,  1.71s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:06<00:00,  1.28s/it]


Grad-CAM visualizations saved for split 4
GPU Memory: 0.00 GB

Generating Grad-CAM visualizations for split 5...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_5\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:09<00:00,  1.65s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [02:03<00:00,  1.35s/it]


Grad-CAM visualizations saved for split 5
GPU Memory: 0.00 GB

Grad-CAM generation complete for all splits!


## Section 6: Results Summary and Analysis

Display comprehensive results and cross-split statistics.

In [9]:
# Display summary of results with GPU memory final state
print("\n" + "="*80)
print("EVALUATION SUMMARY ACROSS ALL SPLITS")
print("="*80)

# Aggregate metrics across splits
aggregated_metrics = {}
for split_key, metrics in results_summary.items():
    split_num = split_key.split('_')[1]
    test_metrics = metrics['test_metrics']
    print(f"\nSplit {split_num} Test Metrics:")
    for metric_name, metric_value in test_metrics.items():
        print(f"  {metric_name}: {metric_value:.4f}")
        if metric_name not in aggregated_metrics:
            aggregated_metrics[metric_name] = []
        aggregated_metrics[metric_name].append(metric_value)

# Calculate mean and std for each metric
print(f"\n{'='*80}")
print("CROSS-SPLIT STATISTICS")
print(f"{'='*80}")
for metric_name, values in aggregated_metrics.items():
    mean_val = np.mean(values)
    std_val = np.std(values)
    print(f"{metric_name}:")
    print(f"  Mean: {mean_val:.4f}")
    print(f"  Std:  {std_val:.4f}")
    print(f"  Min:  {min(values):.4f}")
    print(f"  Max:  {max(values):.4f}")

# Final GPU cleanup
if cuda_available:
    final_gpu_memory = torch.cuda.memory_allocated() / 1e9
    torch.cuda.empty_cache()
    print(f"\n{'='*80}")
    print("CUDA MEMORY CLEANUP")
    print(f"{'='*80}")
    print(f"Final GPU Memory Allocated: {final_gpu_memory:.2f} GB")
    print(f"GPU Cache cleared.")

print(f"\n{'='*80}")
print("Evaluation and visualization complete!")
print(f"Results saved to: {output_base_dir}")
print(f"Execution device: {device}")
print(f"{'='*80}")


EVALUATION SUMMARY ACROSS ALL SPLITS

Split 1 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 2 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 3 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 4 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 5 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

CROSS-SPLIT STATISTICS
accuracy:
  Mean: 0.8200
  Std:  0.0000
  Min:  0.8200
  Max:  0.8200
precision:
  Mean: 0.8100
  Std:  0.0000
  Min:  0.8100
  Max:  0.8100
recall:
  Mean: 0.8200
  Std:  0.0000
  Min:  0.8200
  Max:  0.8200

CUDA MEMORY CLEANUP
Final GPU Memory Allocated: 0.00 GB
GPU Cache cleared.

Evaluation and visualization complete!
Results saved to: C:/Users/Lenovo/Desktop/Lorenzo/Test/res
Execution device: cuda:0
